# **E-Commerce Revenue & Customer Intelligence Project**

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sc

pd.set_option('display.float_format', '{:.2f}'.format)

---

## **1. Project Overview**

**Objective:**
Transform raw transactional data into actionable insights, financial KPIs, customer intelligence, and operational dashboards for strategic decision-making.

**Dataset:**
Merged from six sources:

* **Orders, Order Items, Products, Customers, Payments, Sellers**
* **Final shape:** 115,011 rows × 44 columns

**Key Feature Groups:**

| Group                 | Features                                                                                                                          |
| --------------------- | --------------------------------------------------------------------------------------------------------------------------------- |
| **Order Information** | order_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_customer_date, order_estimated_delivery_date |
| **Financial**         | price, freight_value, payment_value, revenue, cost, profit, freight_ratio                                                         |
| **Customer Info**     | customer_unique_id, customer_city, customer_state, customer_zip_code_prefix                                                       |
| **Product Info**      | product_category_name, product_weight_g, product_length_cm, product_height_cm, product_width_cm                                   |
| **Seller Info**       | seller_city, seller_state                                                                                                         |
| **Time Features**     | Created, year, month, day, weekday, purchase_month, cohort_month, cohort_index                                                    |

---

In [2]:
df = pd.read_csv("df.csv")

In [3]:
df.shape

(115030, 33)

## **2. Step-by-Step Project Flow**

### **Step 1 – Data Integration & Cleaning**

**Actions Taken:**

* Merged datasets: Orders + Order Items + Products + Customers + Payments + Sellers
* Created financial metrics:

  * `revenue = price + freight_value`
  * `cost = 0.8 × price`
  * `profit = revenue − cost`
* Extracted time-based features from `order_purchase_timestamp`
* Removed null rows in key columns (`order_approved_at`, `order_delivered_carrier_date`, `payment_value`)
* Filled missing product dimensions (`length`, `height`, `width`)
* Verified no negative revenue/profit

**Importance:**

* Ensures **clean, reliable analysis**
* Removes incomplete/fake orders
* Builds a **strong foundation for KPIs and analytics**

---

In [4]:
df['revenue'] = df['freight_value'] + df['price']
df['cost'] = 0.8 * df['price'] 
df['profit'] = df['revenue'] - df['cost']

In [5]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

In [6]:
df['year'] = df['order_purchase_timestamp'].dt.year
df['month'] = df['order_purchase_timestamp'].dt.month
df['day'] = df['order_purchase_timestamp'].dt.day
df['weekday'] = df['order_purchase_timestamp'].dt.day_name()

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115030 entries, 0 to 115029
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       115030 non-null  object        
 1   customer_id                    115030 non-null  object        
 2   order_status                   115030 non-null  object        
 3   order_purchase_timestamp       115030 non-null  datetime64[ns]
 4   order_approved_at              115015 non-null  datetime64[ns]
 5   order_delivered_carrier_date   115029 non-null  datetime64[ns]
 6   order_delivered_customer_date  115030 non-null  datetime64[ns]
 7   order_estimated_delivery_date  115030 non-null  datetime64[ns]
 8   order_item_id                  115030 non-null  int64         
 9   product_id                     115030 non-null  object        
 10  seller_id                      115030 non-null  object        
 11  

In [8]:
df.shape

(115030, 40)

In [9]:
df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date        1
order_delivered_customer_date       0
order_estimated_delivery_date       0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
product_category_name               0
product_name_lenght              1628
product_description_lenght       1628
product_photos_qty               1628
product_weight_g                    0
product_length_cm                  20
product_height_cm                  20
product_width_cm                   20
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
payment_sequ

In [10]:
df = df.dropna(subset=[
    'order_approved_at',
    'order_delivered_carrier_date'
])

In [11]:
for col in ['product_length_cm', 'product_height_cm', 'product_width_cm']:
    df[col] = df[col].fillna(df[col].median())

In [12]:
df[df['payment_value'].isnull()]
df = df[df['payment_value'].notnull()]

In [13]:
print("Negative revenue:", (df['revenue'] < 0).sum())
print("Negative profit:", (df['profit'] < 0).sum())

Negative revenue: 0
Negative profit: 0


### **Step 2 – Executive KPI Overview**

**KPIs Calculated:**

* Total Revenue
* Total Profit
* Profit Margin %
* Total Orders
* Total Customers and Total Unique Customers
* Average Order Value (AOV)
* (Optional extras): Monthly Growth %, Top Category Share, New vs Returning Customers

**Transformations:**

* `Revenue` = sum of revenue
* `Profit` = sum of profit
* `Profit Margin` = Profit ÷ Revenue
* `AOV` = Average revenue per order
* Unique counts for orders and customers

**Purpose:**

* Provides a **10-second snapshot** of business health
* Helps investors and executives **make quick strategic decisions**

---

In [14]:
Total_revenue = df['revenue'].sum()
Total_Profit = df['profit'].sum()
Profit_Margin = (Total_Profit / Total_revenue) *100 
total_orders = df['order_id'].nunique()
total_customers = df['customer_unique_id'].nunique()
avg_order_value = df['revenue'].mean()

In [15]:
# Ensure purchase_month is datetime or string 'YYYY-MM'
df['purchase_month'] = pd.to_datetime(df['order_purchase_timestamp']).dt.to_period('M')

# Aggregate revenue per month
monthly_revenue = df.groupby('purchase_month')['revenue'].sum().reset_index()

# Sort by month
monthly_revenue = monthly_revenue.sort_values('purchase_month')

# Calculate month-over-month growth %
monthly_revenue['monthly_growth_pct'] = monthly_revenue['revenue'].pct_change() * 100

print(monthly_revenue)
monthly_revenue.shape

monthly_revenue.to_csv("monthly_revenue.csv", index=False)
print("CSV saved successfully!")

   purchase_month    revenue  monthly_growth_pct
0         2016-10   48133.30                 NaN
1         2016-12      19.62              -99.96
2         2017-01  136634.49           696304.13
3         2017-02  283420.83              107.43
4         2017-03  440064.65               55.27
5         2017-04  413855.41               -5.96
6         2017-05  610359.02               47.48
7         2017-06  513583.24              -15.86
8         2017-07  606902.69               18.17
9         2017-08  675267.54               11.26
10        2017-09  743330.71               10.08
11        2017-10  785306.52                5.65
12        2017-11 1194738.89               52.14
13        2017-12  874067.75              -26.84
14        2018-01 1120930.48               28.24
15        2018-02 1005497.71              -10.30
16        2018-03 1168529.57               16.21
17        2018-04 1171614.31                0.26
18        2018-05 1170957.30               -0.06
19        2018-06 10

In [16]:
# Aggregate revenue by category
category_revenue = df.groupby('product_category_name')['revenue'].sum().reset_index()

# Sort descending
category_revenue = category_revenue.sort_values('revenue', ascending=False)

# Calculate percentage share
category_revenue['revenue_share_pct'] = (category_revenue['revenue'] / category_revenue['revenue'].sum()) * 100

print(category_revenue.head(10))  # Top 10 categories

category_revenue.to_csv("category_revenue.csv", index=False)
print("CSV saved successfully!")

     product_category_name    revenue  revenue_share_pct
12            beleza_saude 1456026.83               9.04
67      relogios_presentes 1315013.93               8.16
14         cama_mesa_banho 1292422.04               8.02
33           esporte_lazer 1159638.36               7.20
45  informatica_acessorios 1068666.04               6.63
55        moveis_decoracao  922629.26               5.73
73   utilidades_domesticas  798858.27               4.96
27              cool_stuff  719293.84               4.46
9               automotivo  696377.68               4.32
41      ferramentas_jardim  593677.82               3.69
CSV saved successfully!


In [17]:
# First purchase per customer
first_purchase = df.groupby('customer_unique_id')['order_purchase_timestamp'].min().reset_index()
first_purchase.columns = ['customer_unique_id', 'first_purchase_date']

# Merge with original data
df_customers = df.merge(first_purchase, on='customer_unique_id', how='left')

# Create 'Customer Type' column
df_customers['customer_type'] = df_customers.apply(
    lambda x: 'New' if x['order_purchase_timestamp'] == x['first_purchase_date'] else 'Returning', axis=1
)

# Aggregate monthly
new_returning_monthly = df_customers.groupby(['purchase_month', 'customer_type'])['customer_unique_id'].nunique().reset_index()

# Pivot for easier view
new_returning_pivot = new_returning_monthly.pivot(index='purchase_month', columns='customer_type', values='customer_unique_id').fillna(0)

print(new_returning_pivot)

customer_type      New  Returning
purchase_month                   
2016-10         262.00       3.00
2016-12           1.00       0.00
2017-01         715.00      20.00
2017-02        1616.00      20.00
2017-03        2503.00      36.00
2017-04        2256.00      37.00
2017-05        3450.00      80.00
2017-06        3037.00      84.00
2017-07        3752.00     100.00
2017-08        4057.00     109.00
2017-09        4003.00     127.00
2017-10        4329.00     132.00
2017-11        7059.00     198.00
2017-12        5338.00     155.00
2018-01        6842.00     204.00
2018-02        6288.00     227.00
2018-03        6774.00     206.00
2018-04        6582.00     204.00
2018-05        6506.00     231.00
2018-06        5875.00     204.00
2018-07        5946.00     184.00
2018-08        6144.00     195.00


### **Step 3 – Sales & Performance Analysis**


✔ **Monthly Table Has:** Revenue, Profit, Orders, Customers, Margin, MoM growth.                                           
✔ **Category Table Has:** Revenue, Profit, Margin, Orders, AOV, Freight impact.                                            
✔ **State Table Has:** Revenue, Profit, Margin, Orders, Customers, Freight impact

---

### 🎯 How These Three Tables Work Together

| Table    | Answers                     |
| -------- | --------------------------- |
| Monthly  | Are we growing?             |
| Category | What products drive profit? |
| State    | Where should we expand?     |

---


### 1️⃣ MONTHLY REVENUE TREND TABLE

**🔹 Purpose**

* Detect seasonality
* Measure growth
* Identify slow periods
* Support forecasting

**✅ Table Name:** `monthly_revenue_summary`

**📊 Columns That Must Be Inside**

| Column                 | Description                            | Why                                    |
| ---------------------- | -------------------------------------- | -------------------------------------- |
| `purchase_month`       | Format: YYYY-MM                        | Time dimension for trend analysis      |
| `total_revenue`        | Sum of revenue per month               | Shows monthly sales performance        |
| `total_profit`         | Sum of profit per month                | Shows monthly profitability            |
| `total_orders`         | Count of unique `order_id` per month   | Shows transaction volume               |
| `total_customers`      | Unique customer count per month        | Shows customer acquisition             |
| `profit_margin_%`      | `(total_profit / total_revenue) × 100` | Shows efficiency, not just sales       |
| `MoM_revenue_growth_%` | Month-over-Month growth %              | Measures business expansion or decline |

**📌 Final Structure Example**
| purchase_month | total_revenue | total_profit | total_orders | total_customers | profit_margin_% | MoM_growth_% |

---


In [18]:
# # Ensure 'purchase_month' is datetime in YYYY-MM format
# df['purchase_month'] = pd.to_datetime(df['order_purchase_timestamp']).dt.to_period('M')

# # Aggregate metrics per month into a DataFrame
# monthly_summary = df.groupby('purchase_month').agg(
#     total_revenue=('revenue', 'sum'),
#     total_profit=('profit', 'sum'),
#     total_orders=('order_id', 'nunique'),       # Unique orders per month
#     total_customers=('customer_id', 'nunique') # Unique customers per month
# ).reset_index()

# # Calculate profit margin %
# monthly_summary['profit_margin_%'] = (monthly_summary['total_profit'] / monthly_summary['total_revenue']) * 100

# # Sort by month
# monthly_summary = monthly_summary.sort_values('purchase_month')

# # Calculate Month-over-Month growth %
# monthly_summary['MoM_growth_%'] = monthly_summary['total_revenue'].pct_change() * 100

# # Optional: convert 'purchase_month' to string 'YYYY-MM'
# monthly_summary['purchase_month'] = monthly_summary['purchase_month'].astype(str)

# # monthly_summary is now a ready-to-use DataFrame
# print(monthly_summary.shape)
# monthly_summary.head()

In [19]:
# ==========================================================
# 1️⃣ Create Clean Monthly Column
# ==========================================================

df['purchase_month'] = (
    pd.to_datetime(df['order_purchase_timestamp'])
    .dt.to_period('M')
    .dt.to_timestamp()
)

# ==========================================================
# 2️⃣ Aggregate Monthly Metrics
# ==========================================================

monthly_summary = df.groupby('purchase_month').agg(
    total_revenue=('revenue', 'sum'),
    total_profit=('profit', 'sum'),
    total_orders=('order_id', 'nunique'),
    total_customers=('customer_id', 'nunique')
).reset_index()

# ==========================================================
# 3️⃣ Create Continuous Monthly Timeline
# ==========================================================

full_month_range = pd.date_range(
    start=monthly_summary['purchase_month'].min(),
    end=monthly_summary['purchase_month'].max(),
    freq='MS'
)

monthly_summary = (
    monthly_summary
    .set_index('purchase_month')
    .reindex(full_month_range)
    .rename_axis('purchase_month')
    .reset_index()
)

# Fill operational metrics with 0 (NOT growth)
fill_cols = ['total_revenue','total_profit','total_orders','total_customers']
monthly_summary[fill_cols] = monthly_summary[fill_cols].fillna(0)

# ==========================================================
# 4️⃣ Safe Financial Metrics
# ==========================================================

# Profit Margin
monthly_summary['profit_margin_%'] = np.where(
    monthly_summary['total_revenue'] > 0,
    (monthly_summary['total_profit'] / monthly_summary['total_revenue']) * 100,
    np.nan
)

# Average Order Value
monthly_summary['AOV'] = np.where(
    monthly_summary['total_orders'] > 0,
    monthly_summary['total_revenue'] / monthly_summary['total_orders'],
    np.nan
)

# ==========================================================
# 5️⃣ Professional MoM Growth Calculation
# ==========================================================

prev_revenue = monthly_summary['total_revenue'].shift(1)

# Only calculate growth if previous revenue is meaningful
MIN_REVENUE_THRESHOLD = monthly_summary['total_revenue'].median() * 0.05

monthly_summary['MoM_growth_%'] = np.where(
    prev_revenue >= MIN_REVENUE_THRESHOLD,
    ((monthly_summary['total_revenue'] - prev_revenue) / prev_revenue) * 100,
    np.nan
)

# Cap extreme outliers (financial best practice)
monthly_summary['MoM_growth_%'] = monthly_summary['MoM_growth_%'].clip(-200, 300)

# ==========================================================
# 6️⃣ Smoothed Growth (More Reliable Trend)
# ==========================================================

monthly_summary['Revenue_3M_MA'] = (
    monthly_summary['total_revenue']
    .rolling(3)
    .mean()
)

monthly_summary['MoM_growth_3M_avg_%'] = (
    monthly_summary['MoM_growth_%']
    .rolling(3)
    .mean()
)

# ==========================================================
# 7️⃣ Optional: Year-over-Year (Much More Stable KPI)
# ==========================================================

monthly_summary['YoY_growth_%'] = (
    monthly_summary['total_revenue']
    .pct_change(12) * 100
)

# ==========================================================
# 8️⃣ Format Output
# ==========================================================

monthly_summary['purchase_month'] = (
    monthly_summary['purchase_month']
    .dt.strftime('%Y-%m')
)

monthly_summary = monthly_summary.sort_values('purchase_month')

monthly_summary.to_csv("monthly_performance_summary.csv", index=False ,float_format="%.2f")
print("monthly table saved!")

print("Final Shape:", monthly_summary.shape)
monthly_summary.head(23)

monthly table saved!
Final Shape: (23, 11)


,purchase_month,total_revenue,total_profit,total_orders,total_customers,profit_margin_%,AOV,MoM_growth_%,Revenue_3M_MA,MoM_growth_3M_avg_%,YoY_growth_%
0,2016-10,48133.30,14789.48,265.00,265.00,30.73,181.64,NaN,NaN,NaN,NaN
1,2016-11,0.00,0.00,0.00,0.00,NaN,NaN,-100.00,NaN,NaN,NaN
2,2016-12,19.62,10.90,1.00,1.00,55.56,19.62,NaN,16050.97,NaN,NaN
3,2017-01,136634.49,40604.42,748.00,748.00,29.72,182.67,NaN,45551.37,NaN,NaN
4,2017-02,283420.83,87946.99,1641.00,1641.00,31.03,172.71,107.43,140024.98,NaN,NaN
5,2017-03,440064.65,135020.18,2546.00,2546.00,30.68,172.85,55.27,286706.66,NaN,NaN
6,2017-04,413855.41,124929.26,2303.00,2303.00,30.19,179.70,-5.96,379113.63,52.25,NaN
7,2017-05,610359.02,187736.00,3545.00,3545.00,30.76,172.17,47.48,488093.03,32.26,NaN
8,2017-06,513583.24,160050.30,3135.00,3135.00,31.16,163.82,-15.86,512599.22,8.56,NaN
9,2017-07,606902.69,194391.92,3872.00,3872.00,32.03,156.74,18.17,576948.32,16.60,NaN



### 2️⃣ CATEGORY PERFORMANCE TABLE

**🔹 Purpose**

* Identify top-selling categories
* Detect high-margin categories
* Spot weak or loss-making categories
* Support pricing strategy

**✅ Table Name:** `category_performance_summary`

**📊 Columns Required**

| Column                  | Description                       | Why                                |
| ----------------------- | --------------------------------- | ---------------------------------- |
| `product_category_name` | Category identifier               | Identify products                  |
| `total_revenue`         | Sum of revenue by category        | Shows category performance         |
| `total_profit`          | Sum of profit by category         | Shows category profitability       |
| `total_orders`          | Order count per category          | Volume measure                     |
| `total_quantity_sold`   | If available                      | Measures product demand            |
| `profit_margin_%`       | `(profit / revenue) × 100`        | Shows efficiency                   |
| `avg_order_value`       | `revenue / orders`                | Shows category ticket size         |
| `freight_ratio_avg`     | Average `freight_value / revenue` | Detects logistics-heavy categories |

**📌 Final Structure Example**
| category | total_revenue | total_profit | total_orders | profit_margin_% | AOV | freight_ratio |

---

In [20]:
# Aggregate metrics by category
category_summary = df.groupby('product_category_name').agg(
    total_revenue=('revenue', 'sum'),
    total_profit=('profit', 'sum'),
    total_orders=('order_id', 'nunique'),
    total_freight=('freight_value', 'sum')
).reset_index()

# Calculate profit margin
category_summary['profit_margin_%'] = (category_summary['total_profit'] / category_summary['total_revenue']) * 100

# Calculate Average Order Value
category_summary['avg_order_value'] = category_summary['total_revenue'] / category_summary['total_orders']

# Calculate Freight Ratio
category_summary['freight_ratio_avg'] = category_summary['total_freight'] / category_summary['total_revenue']

# Sort by total revenue descending
category_summary = category_summary.sort_values('total_revenue', ascending=False)

# Select final columns
category_summary = category_summary[
    ['product_category_name', 'total_revenue', 'total_profit', 'total_orders', 
     'profit_margin_%', 'avg_order_value', 'freight_ratio_avg']
]

# Save CSV
category_summary.to_csv("category_performance_summary.csv", index=False)
print("Category table saved!")
print(category_summary.shape)
category_summary.head(10)

Category table saved!
(74, 7)


,product_category_name,total_revenue,total_profit,total_orders,profit_margin_%,avg_order_value,freight_ratio_avg
12,beleza_saude,1456026.83,438960.20,8646,30.15,168.40,0.13
67,relogios_presentes,1315013.93,344483.69,5493,26.20,239.40,0.08
14,cama_mesa_banho,1292422.04,430154.73,9271,33.28,139.40,0.17
33,esporte_lazer,1159638.36,367321.37,7527,31.68,154.06,0.15
45,informatica_acessorios,1068666.04,333595.74,6529,31.22,163.68,0.14
55,moveis_decoracao,922629.26,326030.22,6303,35.34,146.38,0.19
73,utilidades_domesticas,798858.27,280308.08,5743,35.09,139.10,0.19
27,cool_stuff,719293.84,211962.82,3556,29.47,202.28,0.12
9,automotivo,696377.68,214072.28,3809,30.74,182.82,0.13
41,ferramentas_jardim,593677.82,199873.76,3447,33.67,172.23,0.17


### 3️⃣ STATE-WISE PERFORMANCE TABLE

**🔹 Purpose**

* Guide regional expansion
* Detect low-margin states
* Optimize logistics
* Plan regional marketing

**✅ Table Name:** `state_performance_summary`

**📊 Columns Required**

| Column              | Description                       | Why                      |
| ------------------- | --------------------------------- | ------------------------ |
| `customer_state`    | Geographic identifier             | State-level analysis     |
| `total_revenue`     | Sum revenue by state              | Revenue performance      |
| `total_profit`      | Sum profit by state               | Profitability measure    |
| `total_orders`      | Order volume per state            | Transaction volume       |
| `total_customers`   | Unique customers per state        | Customer reach           |
| `profit_margin_%`   | `(profit / revenue) × 100`        | Efficiency check         |
| `avg_order_value`   | `revenue / orders`                | Shows average order size |
| `avg_freight_ratio` | Average `freight_value / revenue` | Logistics efficiency     |

**📌 Final Structure Example**
| state | total_revenue | total_profit | total_orders | total_customers | profit_margin_% | AOV | freight_ratio |

---


In [21]:
# Aggregate metrics by state
state_summary = df.groupby('customer_state').agg(
    total_revenue=('revenue', 'sum'),
    total_profit=('profit', 'sum'),
    total_orders=('order_id', 'nunique'),
    total_customers=('customer_unique_id', 'nunique'),
    total_freight=('freight_value', 'sum')
).reset_index()

# Calculate Profit Margin %
state_summary['profit_margin_%'] = (
    state_summary['total_profit'] / state_summary['total_revenue']
) * 100

# Calculate Average Order Value (AOV)
state_summary['AOV'] = (
    state_summary['total_revenue'] / state_summary['total_orders']
)

# Calculate Freight Ratio
state_summary['freight_ratio'] = (
    state_summary['total_freight'] / state_summary['total_revenue']
)

# Sort by revenue (highest performing states first)
state_summary = state_summary.sort_values('total_revenue', ascending=False)

# Select final structured columns
state_summary = state_summary[
    [
        'customer_state',
        'total_revenue',
        'total_profit',
        'total_orders',
        'total_customers',
        'profit_margin_%',
        'AOV',
        'freight_ratio'
    ]
]

# Save to CSV
state_summary.to_csv("state_performance_summary.csv", index=False)

print("State performance table created successfully!")
print(state_summary.shape)
state_summary.head(10)

State performance table created successfully!
(27, 8)


,customer_state,total_revenue,total_profit,total_orders,total_customers,profit_margin_%,AOV,freight_ratio
25,SP,6041601.93,1796896.11,40487,39143,29.74,149.22,0.12
18,RJ,2158350.99,682220.40,12348,11915,31.61,174.79,0.15
10,MG,1882713.67,597878.77,11351,10998,31.76,165.86,0.15
22,RS,903767.62,291509.87,5342,5165,32.25,169.18,0.15
17,PR,809138.71,258163.97,4923,4769,31.91,164.36,0.15
4,BA,624810.85,208056.95,3256,3158,33.30,191.90,0.17
23,SC,614303.48,195332.09,3546,3449,31.80,173.24,0.15
6,DF,358052.81,112493.11,2080,2019,31.42,172.14,0.14
8,GO,351448.53,113062.07,1957,1895,32.17,179.59,0.15
7,ES,328146.52,106143.26,1995,1928,32.35,164.48,0.15




### 🔶 STEP 4 – Customer Intelligence (RFM Model)

## 🎯 Purpose

Move from **sales analysis → customer value analysis**.
Identify:

* Best customers
* Churn risk customers
* Revenue-driving segments
* Retention opportunities

---

# ✅ Required Features

Only 4 columns needed:

* `customer_unique_id`
* `order_purchase_timestamp`
* `order_id`
* `revenue`

---

# ✅ RFM Metrics (Core Logic)

### 🔹 1️⃣ Recency (R)

Days since last purchase.

* Lower value → Active customer
* Higher value → Churn risk

**Why:** Recent buyers are more likely to buy again.

---

### 🔹 2️⃣ Frequency (F)

Number of unique orders per customer.

* Higher → Loyal customer
* Lower → One-time buyer

**Why:** Repeat buyers create stable revenue.

---

### 🔹 3️⃣ Monetary (M)

Total revenue per customer.

* Higher → High-value customer
* Lower → Low contribution

**Why:** Shows true financial impact.

---

# ✅ RFM Scoring (1–5)

Convert each metric into scores:

* Recency → Lower = Higher score
* Frequency → Higher = Higher score
* Monetary → Higher = Higher score

Create final:

```
RFM Score = R + F + M
```

---

# ✅ Customer Segments (Simple Structure)

| Segment      | Meaning                 |
| ------------ | ----------------------- |
| 🏆 Champions | High R, F, M            |
| 💎 Loyal     | High F & M              |
| 🙂 Regular   | Moderate                |
| ⚠ At Risk    | High Recency (inactive) |
| ❌ Lost       | Very low engagement     |

---

# ✅ What RFM Tells You

✔ Who generates most revenue
✔ Who may churn soon
✔ Where to spend marketing budget
✔ Which customers need retention campaigns
✔ Foundation for CLV calculation

---

# ✅ Business Impact

* Improves retention strategy
* Reduces churn
* Increases marketing ROI
* Enables personalized targeting
* Converts dashboard → Decision Intelligence System

---

# 🔥 One-Line Summary

RFM transforms customer data into actionable business segments for growth, retention, and profitability optimization.

---


In [22]:
analysis_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

In [23]:
rfm = df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (analysis_date - x.max()).days,  # Recency
    'order_id': 'nunique',                                                  # Frequency
    'revenue': 'sum'                                                        # Monetary
}).reset_index()

rfm.columns = ['customer_unique_id', 'Recency', 'Frequency', 'Monetary']

In [24]:
# Recency Score (lower recency = better score)
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])

# Frequency Score
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])

# Monetary Score
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

In [25]:
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

rfm['RFM_Score'] = rfm['R_score'].astype(str) + \
                   rfm['F_score'].astype(str) + \
                   rfm['M_score'].astype(str)

In [26]:
rfm['RFM_Score'].shape

(93335,)

In [27]:
def segment_customer(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Champions'
    elif row['F_score'] >= 4 and row['M_score'] >= 3:
        return 'Loyal'
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'
    else:
        return 'Regular'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

In [28]:
segment_summary = rfm.groupby('Segment').agg({
    'customer_unique_id': 'count',
    'Monetary': 'sum',
    'Frequency': 'mean'
}).reset_index()

segment_summary.columns = ['Segment', 'Customer_Count', 'Total_Revenue', 'Avg_Frequency']

print(segment_summary)

     Segment  Customer_Count  Total_Revenue  Avg_Frequency
0    At Risk           13295     1594408.72           1.01
1  Champions            6473     2116124.48           1.18
2      Loyal           16440     3757412.94           1.11
3    Regular           57127     8642545.42           1.00


### **Step 5 – Customer & Profit Intelligence**

1. **Revenue by Segment:** Focus marketing on high-value customers
2. **Pareto Analysis:** Check if 20% of customers generate 80% revenue
3. **Cohort Retention Analysis:** Understand when customers stop buying
4. **Profitability Deep Dive (Category Level):** Detect margin leakages
5. **Customer Lifetime Value (CLV):** Determine customer worth and set acquisition budget
6. **Churn Risk:** Recency > 180 days → high risk, plan win-back campaigns

---



# 🔥 What You Now Know

✔ Who generates money
✔ Where revenue risk exists
✔ When customers leave
✔ Which categories hurt margin
✔ How much customers are worth
✔ Who needs retention action

---

## 1️⃣ Revenue by Segment

**Features Used:**
Segment, Monetary

**Observation Goal:**
Which customer group drives most revenue?

**Meaning:**

* High revenue from Champions/Loyal → Protect & reward
* Revenue mostly from Regular → Improve retention

**Decision Impact:**
Marketing budget allocation

---

In [29]:
segment_summary = rfm.groupby('Segment').agg({
    'customer_unique_id': 'count',
    'Monetary': 'sum',
    'Frequency': 'mean'
}).reset_index()

segment_summary.columns = ['Segment', 'Customer_Count', 'Total_Revenue', 'Avg_Frequency']

monthly_summary.to_csv("segment_summary.csv", index=False ,float_format="%.2f")
print("segment summary table saved!")

print(segment_summary)

segment summary table saved!
     Segment  Customer_Count  Total_Revenue  Avg_Frequency
0    At Risk           13295     1594408.72           1.01
1  Champions            6473     2116124.48           1.18
2      Loyal           16440     3757412.94           1.11
3    Regular           57127     8642545.42           1.00




## 2️⃣ Pareto Analysis (80/20 Rule)

**Features Used:**
Customer ID, Monetary

**Observation Goal:**
Do 20% customers generate 80% revenue?

**Meaning:**

* Yes → High dependency risk
* No → Revenue distributed & stable

**Decision Impact:**
Revenue risk management

---

In [30]:
m = df.groupby('customer_unique_id').agg({
    'revenue': 'sum'                                                        # Monetary
}).reset_index()
m.columns=['customer_unique_id','Monetary']

m = m.sort_values(by='Monetary', ascending=False)
m.head()



,customer_unique_id,Monetary
3724,0a0a92112bd4c708ca5fde585afaa872,13664.08
71801,c4b224d2c784bae11ae98b6ae9f2454c,11111.40
48724,85963fd37bfd387aa6d915d8a1065486,10553.28
69541,be74c431147c32ab2d7c7cef5e4a995f,10055.22
86868,edf81e1f3070b9dac83ec83dacdbb9bc,8389.52


In [31]:
total_revenue = m['Monetary'].sum()
len20 = int(len(m) * 0.20)
top_20_customers = m.head(len20)
top_20_revenue = top_20_customers['Monetary'].sum()
ratio = top_20_revenue / total_revenue
print(ratio)

0.5455222981414728




## 3️⃣ Cohort Retention Analysis

**Features Used:**
Customer ID, Purchase Month, Cohort Month

**Observation Goal:**
When do customers stop buying?

**Meaning:**

* Drop after Month 1 → Weak onboarding
* Stable retention → Strong loyalty

**Decision Impact:**
Retention campaign timing

---

In [32]:
df['Cohort_Month'] = df.groupby('customer_unique_id')['purchase_month'].transform('min')
df['Cohort_Index'] = (
    (df['purchase_month'].dt.year - df['Cohort_Month'].dt.year) * 12 +
    (df['purchase_month'].dt.month - df['Cohort_Month'].dt.month)
)


In [33]:
df['Cohort_Index'].head()

0    1
1    1
2    1
3    0
4    0
Name: Cohort_Index, dtype: int32

In [34]:
cohort_data = df.groupby(['Cohort_Month', 'Cohort_Index'])['customer_unique_id'].nunique().reset_index()

In [35]:
cohort_pivot = cohort_data.pivot(index='Cohort_Month',
                                 columns='Cohort_Index',
                                 values='customer_unique_id')

cohort_pivot.head()

Cohort_Index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
Cohort_Month,,,,,,,,,,,,,,,,,,,,
2016-10-01,262.00,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,1.00,NaN,1.00,NaN,1.00,NaN,1.00,NaN,1.00,2.00,2.00
2016-12-01,1.00,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01,715.00,2.00,2.00,1.00,3.00,1.00,3.00,1.00,1.00,NaN,3.00,1.00,5.00,3.00,1.00,1.00,2.00,3.00,1.00,NaN
2017-02-01,1616.00,3.00,5.00,2.00,7.00,2.00,4.00,3.00,1.00,3.00,2.00,5.00,2.00,3.00,2.00,1.00,1.00,3.00,NaN,NaN
2017-03-01,2503.00,11.00,9.00,10.00,9.00,4.00,4.00,8.00,8.00,2.00,9.00,3.00,5.00,3.00,4.00,6.00,2.00,3.00,NaN,NaN


In [36]:
retention = cohort_pivot.divide(cohort_pivot[0], axis=0)
retention.head()

Cohort_Index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
Cohort_Month,,,,,,,,,,,,,,,,,,,,
2016-10-01,1.00,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,0.01,0.01
2016-12-01,1.00,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,0.00,0.00,0.01,0.00,0.00,0.00,0.00,0.00,0.00,NaN
2017-02-01,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN
2017-03-01,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN




## 4️⃣ Category Profitability Deep Dive

**Features Used:**
Product Category, Revenue, Profit

**Observation Goal:**
Which categories leak margin?

**Meaning:**

* High revenue + low profit → Margin issue
* Low revenue + high profit → Growth opportunity

**Decision Impact:**
Pricing & inventory strategy

---

In [37]:
category_profit = df.groupby('product_category_name').agg({
    'revenue': 'sum',
    'profit': 'sum'
}).reset_index()

In [38]:
category_profit['Profit_Margin_%'] = (
    category_profit['profit'] / category_profit['revenue']
) * 100

category_profit = category_profit.sort_values(
    by='revenue',
    ascending=False
)

In [39]:
print(category_profit.shape)
category_profit.head(10)

(74, 4)


,product_category_name,revenue,profit,Profit_Margin_%
12,beleza_saude,1456026.83,438960.20,30.15
67,relogios_presentes,1315013.93,344483.69,26.20
14,cama_mesa_banho,1292422.04,430154.73,33.28
33,esporte_lazer,1159638.36,367321.37,31.68
45,informatica_acessorios,1068666.04,333595.74,31.22
55,moveis_decoracao,922629.26,326030.22,35.34
73,utilidades_domesticas,798858.27,280308.08,35.09
27,cool_stuff,719293.84,211962.82,29.47
9,automotivo,696377.68,214072.28,30.74
41,ferramentas_jardim,593677.82,199873.76,33.67




## 5️⃣ Customer Lifetime Value (CLV)

**Features Used:**
Customer ID, Monetary, Frequency

**Observation Goal:**
How much is each customer worth?

**Meaning:**

* High CLV → Invest in retention
* Low CLV → Low acquisition spend

**Decision Impact:**
Set customer acquisition cost (CAC) limit

---


In [40]:
clv = df.groupby('customer_unique_id').agg({
    'revenue': 'sum',        # Total revenue (Monetary)
    'order_id': 'nunique',    # Total orders (Frequency)
    'order_purchase_timestamp' : ['min', 'max']
}).reset_index()

clv.columns = [
    'customer_unique_id',
    'Monetary',
    'Frequency',
    'First_Purchase',
    'Last_Purchase'
]

clv['Lifespan_Months'] = (
    (clv['Last_Purchase'].dt.year - clv['First_Purchase'].dt.year) * 12 +
    (clv['Last_Purchase'].dt.month - clv['First_Purchase'].dt.month)
)
clv['Lifespan_Months'] = clv['Lifespan_Months'] + 1
clv['AOV'] = clv['Monetary'] / clv['Frequency']
clv['CLV'] = clv['AOV'] * clv['Frequency'] * clv['Lifespan_Months']

clv = clv.sort_values(by='CLV', ascending=False)

clv.head()

,customer_unique_id,Monetary,Frequency,First_Purchase,Last_Purchase,Lifespan_Months,AOV,CLV
29107,4facc2e6fbc2bffab2fea92d2b4aa7e4,1760.75,4,2017-06-18 23:14:46,2018-08-13 22:51:49,15,440.19,26411.25
32876,59d66d72939bc9497e19d89c61a96d5f,3559.99,2,2017-03-02 12:13:18,2017-08-10 22:09:50,6,1780.00,21359.94
75507,cef29e793e232d30250331804cdb7000,1906.68,3,2017-03-09 18:15:38,2018-01-18 19:59:07,11,635.56,20973.48
10903,1da09dd64e235e7c2f29a4faff33535c,2164.40,3,2017-05-10 14:04:15,2018-01-11 11:16:49,9,721.47,19479.60
26824,495fe7ae31686381c8145b2936e761dc,1382.60,2,2017-08-30 14:16:14,2018-08-05 18:25:52,13,691.30,17973.80


In [41]:
average_clv = clv['CLV'].mean()
print(average_clv)

202.350192853699



## 6️⃣ Churn Risk Identification

**Features Used:**
Recency

**Observation Goal:**
Who hasn’t purchased in 180+ days?

**Meaning:**

* High recency → Churn risk
* Low recency → Active customer

**Decision Impact:**
Win-back campaigns

---

In [42]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

reference_date = df['order_purchase_timestamp'].max()

In [43]:
recency_df = df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': 'max'
}).reset_index()

recency_df.columns = ['customer_unique_id', 'Last_Purchase']

recency_df['Recency_Days'] = (
    reference_date - recency_df['Last_Purchase']
).dt.days

In [44]:
recency_df['Churn_Risk'] = recency_df['Recency_Days'].apply(
    lambda x: 'High Risk' if x > 180 else 'Active'
)

In [45]:
churn_rate = (recency_df['Churn_Risk'] == 'High Risk').mean() * 100
print(f"Churn Rate: {churn_rate:.2f}%")

Churn Rate: 58.92%


## **3. Final Outcome**

* **Full E-commerce Data Warehouse**
* **Financial KPI System**
* **Customer Intelligence Model**
* **Retention Analytics**
* **Profit Optimization Framework**
* **Executive Dashboard Structure**

**One-line Summary:**
From **raw transactional data → a complete Business Intelligence & Decision Support System.**

---

In [46]:
df.to_csv("df_finale.csv", index=False)
print("CSV saved successfully!")

CSV saved successfully!
